# 05 · Affect and narrator-centred events

**Spatial Humanities 2026 workshop**

This notebook examines sentiment, emotion and event extraction as **analytical annotations** rather than direct readings of human interior states.

## Learning goals
- compare transparent rule baselines with optional contextual/HF models;
- inspect exactly which lexical cues drive a rule result;
- identify domain assumptions and false positives;
- extract narrator-centred movement/action events;
- build a simple narrative-sequence view;
- distinguish computational affect labels from psychological ground truth.

> **Key message:** Richer annotation creates richer questions and richer risks.

In [ ]:
# Independent Colab setup.
import os, sys, json, subprocess, pathlib

REPO = "https://github.com/IgnatiusEzeani/spatio-textual.git"
BRANCH = "spatial-humanities-2026"

if not pathlib.Path("spatio-textual").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "spatio-textual"], check=True)
os.chdir("spatio-textual")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

FAST_MODE = True
print("Ready. FAST_MODE =", FAST_MODE)

## 1. Public-safe teaching passages

These are instructor-created examples. They are designed to stress the methods without reproducing controlled testimony material.

In [ ]:
segments = [
    "We reached the village at sunset and felt relieved to find shelter.",
    "The road was quiet and we waited beside the river.",
    "I was afraid as we crossed the dark forest, but later I felt safe.",
    "We left the village and walked to the station before returning home.",
    "The children returned happily from summer camp.",
]

for i, text in enumerate(segments):
    print(i, text)

## 2. Transparent rule-based sentiment

Rule baselines are valuable because we can inspect their assumptions directly. That transparency does not make the assumptions neutral.

In [ ]:
from spatio_textual.sentiment import SentimentAnalyzer, POSITIVE, NEGATIVE, _tokens

sent_rule = SentimentAnalyzer(backend="rule")
sent_results = sent_rule.predict(segments)

def sentiment_evidence(text):
    toks = _tokens(text)
    return {
        "positive_terms": [t for t in toks if t in POSITIVE],
        "negative_terms": [t for t in toks if t in NEGATIVE],
    }

import pandas as pd

sent_table = []
for text, result in zip(segments, sent_results):
    sent_table.append({
        "text": text,
        "label": result["label"],
        "score": result["score"],
        **sentiment_evidence(text),
        "distribution": result["distribution"],
    })

display(pd.DataFrame(sent_table))

### Audit the result, not just the label

Look especially at the final sentence. The current domain-oriented lexicon contains **camp** as a negative cue. In a different domain, such as a summer-camp sentence, the same rule can become misleading.

This is a useful example of **domain assumption leakage**:

> a transparent rule can be easy to audit and still encode a theory or corpus-specific expectation.

Transparency helps us *find* the problem. It does not automatically solve it.

## 3. Rule-based emotion

The current rule backend uses an Ekman-style label inventory. We expose matched terms so you can inspect why a label was produced.

In [ ]:
from spatio_textual.emotion import EmotionAnalyzer, LEXICON, _tokens as emotion_tokens

emo_rule = EmotionAnalyzer(backend="rule")
emo_results = emo_rule.predict(segments)

def emotion_evidence(text):
    toks = emotion_tokens(text)
    return {
        emotion: [t for t in toks if t in words]
        for emotion, words in LEXICON.items()
        if any(t in words for t in toks)
    }

emo_table = []
for text, result in zip(segments, emo_results):
    emo_table.append({
        "text": text,
        "label": result["label"],
        "score": result["score"],
        "matched_terms": emotion_evidence(text),
        "distribution": result["distribution"],
    })

display(pd.DataFrame(emo_table))

### Interpretive caution

These outputs are best treated as:
- reproducible computational descriptors;
- prompts for corpus exploration;
- variables whose distribution can be compared across narrative segments.

They are **not** direct measurements of a narrator's psychological state.

Label choice, taxonomy, segmentation, lexical resources and training data all affect the result.

## 4. Optional contextual classifiers

If you have a suitable runtime and want a live comparison, switch `FAST_MODE = False`.

The workshop's default path does not require downloading transformer models. Release versions will include precomputed fallback outputs generated from documented model/version combinations.

In [ ]:
if not FAST_MODE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-transformers.txt"], check=True)

    hf_sent = SentimentAnalyzer(backend="hf")
    hf_emo = EmotionAnalyzer(backend="hf")

    hf_sent_results = hf_sent.predict(segments[:3])
    hf_emo_results = hf_emo.predict(segments[:3])

    display(pd.DataFrame([
        {
            "text": text,
            "rule_sentiment": sent_results[i]["label"],
            "hf_sentiment": hf_sent_results[i]["label"],
            "rule_emotion": emo_results[i]["label"],
            "hf_emotion": hf_emo_results[i]["label"],
        }
        for i, text in enumerate(segments[:3])
    ]))
else:
    print("FAST_MODE=True: live HF models skipped.")

## 5. Narrator-centred events

Entity extraction tells us *what is mentioned*. Event extraction begins to ask *what the narrator does or experiences*.

We keep event evidence tied to the source text and avoid turning every verb into a historical claim.

In [ ]:
from spatio_textual.utils import load_spacy_model, Annotator

narrative = (
    "I left the village and walked to the river. "
    "We waited there until evening. "
    "Then we returned home."
)

nlp = load_spacy_model("en_core_web_sm", add_entity_ruler=True)
annotator = Annotator(nlp=nlp, model_name="en_core_web_sm", link_places=False)
event_record = annotator.annotate(
    narrative,
    include_entities=True,
    include_verbs=True,
    include_events=True,
    include_text=True,
)

print("Narrative:", narrative)
print("\nNarrator-centred events:")
display(pd.DataFrame(event_record["event_data"]))

If `en_core_web_sm` is not installed, the package falls back safely to a lightweight pipeline. The fallback can still identify some motion/experience cues, but dependency-based subject interpretation will be weaker. That runtime distinction should be recorded in telemetry.

In [ ]:
print("Telemetry:")
print(json.dumps(event_record["telemetry"], indent=2))

## 6. Narrative sequence, not a diagnosis

A simple plot can show how a classifier's outputs change across segments. The y-axis below represents the rule model's **negative probability**, not a person's emotional intensity.

In [ ]:
import matplotlib.pyplot as plt

negative_probability = [r["distribution"]["negative"] for r in sent_results]

plt.figure(figsize=(9, 4))
plt.plot(range(len(segments)), negative_probability, marker="o")
plt.xticks(range(len(segments)), [f"S{i}" for i in range(len(segments))])
plt.xlabel("Narrative segment")
plt.ylabel("Rule-model negative probability")
plt.title("Computational affect signal across narrative sequence")
plt.ylim(0, 1)
plt.show()

## 7. Error-analysis exercise

For each output, ask:

1. **Selection:** did the system analyse the right textual unit?
2. **Ontology:** are these sentiment/emotion labels appropriate for the research question?
3. **Evidence:** which words or contextual features drove the result?
4. **Domain:** is the model carrying assumptions from another corpus?
5. **Interpretation:** what scholarly claim would be justified, and what would overreach?

Suggested challenge: rewrite one sentence so that the *human interpretation* remains similar while the rule result changes substantially.

In [ ]:
challenge = {
    "original": "The children returned happily from summer camp.",
    "rewrite":  "The children came back cheerful after their summer programme.",
}

for label, text in challenge.items():
    s = sent_rule.predict([text])[0]
    e = emo_rule.predict([text])[0]
    print(label, "=>", {
        "sentiment": s["label"],
        "emotion": e["label"],
        "sentiment_evidence": sentiment_evidence(text),
        "emotion_evidence": emotion_evidence(text),
    })

## 8. Take-away

Affect and event extraction can enrich spatial narratives by connecting:

**where → what happened → how the passage is computationally characterised**

But every added layer introduces assumptions about segmentation, ontology, domain, evidence and interpretation.

**Next:** evidence-first LLM structured extraction, where we explicitly separate model proposal from source-grounded validation.